In [ ]:
import time
start_time = time.time()

import math
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 160)

In [ ]:
device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
print(f'Selected device: {device}')

model_name = 'textattack/distilbert-base-uncased-MRPC'
dataset_name = 'glue'
dataset_config = 'mrpc'
split_name = 'validation'
batch_size = 64
max_length = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

id2label = model.config.id2label if hasattr(model.config, 'id2label') and model.config.id2label else {0: 'LABEL_0', 1: 'LABEL_1'}

print(f'Loaded model: {model_name}')
print(f'Batch size: {batch_size}')
print(f'Max length: {max_length}')
print(f'Labels: {id2label}')

In [ ]:
dataset = load_dataset(dataset_name, dataset_config, split=split_name)
print(f'Dataset: {dataset_name}/{dataset_config} | Split: {split_name}')
print(f'Number of examples: {len(dataset)}')

preview_df = dataset.select(range(min(5, len(dataset)))).to_pandas()[['sentence1', 'sentence2', 'label']].copy()
print(preview_df.to_string(index=False))

In [ ]:
all_rows = []
predictions = []
true_labels = []

for start_idx in range(0, len(dataset), batch_size):
    end_idx = min(start_idx + batch_size, len(dataset))
    batch = dataset[start_idx:end_idx]

    inputs = tokenizer(
        batch['sentence1'],
        batch['sentence2'],
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors='pt'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)

    logits_cpu = logits.detach().cpu().numpy()
    probs_cpu = probs.detach().cpu().numpy()
    preds_cpu = preds.detach().cpu().tolist()
    labels_cpu = list(batch['label'])

    predictions.extend(preds_cpu)
    true_labels.extend(labels_cpu)

    for i in range(len(labels_cpu)):
        true_label = int(labels_cpu[i])
        predicted_label = int(preds_cpu[i])
        row = {
            'example_index': start_idx + i,
            'sentence1': batch['sentence1'][i],
            'sentence2': batch['sentence2'][i],
            'true_label': true_label,
            'predicted_label': predicted_label,
            'true_label_name': id2label.get(true_label, str(true_label)),
            'predicted_label_name': id2label.get(predicted_label, str(predicted_label)),
            'correct': true_label == predicted_label,
            'logit_label_0': float(logits_cpu[i][0]),
            'logit_label_1': float(logits_cpu[i][1]),
            'prob_label_0': float(probs_cpu[i][0]),
            'prob_label_1': float(probs_cpu[i][1]),
            'predicted_confidence': float(probs_cpu[i][predicted_label])
        }
        all_rows.append(row)

print(f'Completed inference for {len(predictions)} examples.')

In [ ]:
predictions_df = pd.DataFrame(all_rows)

accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average='binary',
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions, labels=[0, 1])

results_df = pd.DataFrame([
    {
        'model_name': model_name,
        'dataset': f'{dataset_name}/{dataset_config}',
        'split': split_name,
        'num_examples': len(dataset),
        'batch_size': batch_size,
        'max_length': max_length,
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'device': str(device)
    }
])

print(results_df.to_string(index=False))
print('Confusion Matrix [[tn, fp], [fn, tp]]:')
print(cm)

In [ ]:
print(predictions_df.head(10).to_string(index=False))

mismatches_df = predictions_df[~predictions_df['correct']].copy()
print(f'\nTotal mismatches: {len(mismatches_df)}')
if len(mismatches_df) > 0:
    print(mismatches_df[['example_index', 'sentence1', 'sentence2', 'true_label', 'predicted_label', 'logit_label_0', 'logit_label_1', 'prob_label_0', 'prob_label_1', 'predicted_confidence']].head(10).to_string(index=False))

In [ ]:
label_summary_df = predictions_df.groupby(['true_label', 'predicted_label']).size().reset_index(name='count')
confidence_summary_df = predictions_df.groupby('correct')['predicted_confidence'].agg(['count', 'mean', 'min', 'max']).reset_index()

print('Per-label prediction counts:')
print(label_summary_df.to_string(index=False))
print('\nConfidence summary by correctness:')
print(confidence_summary_df.to_string(index=False))

In [ ]:
output_predictions_path = 'mrpc_validation_predictions_detailed.csv'
output_metrics_path = 'mrpc_validation_metrics.csv'

predictions_df.to_csv(output_predictions_path, index=False)
results_df.to_csv(output_metrics_path, index=False)

print(f'Saved detailed predictions to: {output_predictions_path}')
print(f'Saved metrics to: {output_metrics_path}')

elapsed_seconds = time.time() - start_time
print(f'Total runtime (seconds): {elapsed_seconds:.2f}')